In [10]:
from datetime import datetime, timezone
from funciones_base import epoch_medio
from catalogo import leer_satelites, contar_por_grupo, lanzamientos_por_anio
from satelite import crear_satelites
from analisis import contar_por_tipo
from analisis import mas_bajo, mas_alto, top_n, resumen_por_grupo, en_zona, ordenar_por

#1 Carga de datos:
Leo el archivo y calculo dónde está cada satélite ahora.

In [11]:
registros = leer_satelites("datos/satelites.json")
ahora = datetime.now(timezone.utc)
satelites = crear_satelites(registros, ahora)

fecha_datos = epoch_medio(registros)
print("Satélites en el archivo:", len(registros))
print("Fecha media de los datos:", fecha_datos)
print("Días transcurridos:", (ahora - fecha_datos).days)
print("Descartados por crear_satelites:", len(registros) - len(satelites))

Satélites en el archivo: 16710
Fecha media de los datos: 2026-10-14 06:30:18.482258+00:00
Días transcurridos: 16
Descartados por crear_satelites: 35


El archivo tiene 16710 satélites. Los datos son de 14 de octubre de 2026 (la fecha media es 2026-10-14, y pasaron 14 días) y se descartaron 23.

#2 Grupos:
Cuento cuántos satélites hay en cada grupo.

In [12]:
conteo = contar_por_grupo(registros)
print(conteo)
porcentaje = conteo["Starlink"] / len(registros) * 100
print("Porcentaje de Starlink:", round(porcentaje, 1), "%")

{'Otros': 4075, 'GPS': 40, 'Globalstar': 36, 'BeiDou': 55, 'Iridium': 80, 'OneWeb': 651, 'Starlink': 11125, 'Qianfan': 257, 'Kuiper': 391}
Porcentaje de Starlink: 66.6 %


Starlink es el más grande, con 66.6% del total.

#3 Lanzamientos por año
Cuántos satélites activos se lanzaron cada año desde 2015.

In [13]:
por_anio = lanzamientos_por_anio(registros)
desde_2015 = {}
for anio in sorted(por_anio):
    if anio >= 2015:
        desde_2015[anio] = por_anio[anio]
print(desde_2015)
mejor = max(desde_2015, key=desde_2015.get)
print("Año con más lanzamientos:", mejor)

{2015: 96, 2016: 83, 2017: 136, 2018: 194, 2019: 145, 2020: 360, 2021: 994, 2022: 1718, 2023: 2266, 2024: 2429, 2025: 4289, 2026: 3285}
Año con más lanzamientos: 2025


El año con más lanzamientos fue 2025. Es una estimación, porque el archivo solo tiene los satélites activos hoy.

#4 Tipos de órbita:
Cuento los satélites según su órbita.

In [18]:
print(contar_por_tipo(satelites))

{'LEO': 15865, 'MEO': 196, 'GEO': 587, 'HEO': 27}


La órbita LEO domina, con 15865 satélites.

#5 Extremos:
El satélite más bajo, el más alto y el top 10 de cada uno.

In [15]:
print("Más bajo:", mas_bajo(satelites))
print("Más alto:", mas_alto(satelites))

print("\nTop 10 más altos:")
for s in top_n(satelites, 10):
    print(s)

print("\nTop 10 más bajos:")
for s in top_n(satelites, 10, mas_altos=False):
    print(s)

Más bajo: STARLINK-30236 | LEO | 110.2 km
Más alto: TESS | HEO | 283289.8 km

Top 10 más altos:
TESS | HEO | 283289.8 km
MMS 2 | HEO | 142850.0 km
MMS 1 | HEO | 142818.6 km
MMS 3 | HEO | 142811.7 km
MMS 4 | HEO | 142793.2 km
SMILE | HEO | 106056.9 km
CXO | HEO | 95436.6 km
XMM-NEWTON | HEO | 87198.6 km
THEMIS E | HEO | 74088.7 km
THEMIS D | HEO | 70483.9 km

Top 10 más bajos:
STARLINK-30236 | LEO | 110.2 km
COSMOS 2572 | LEO | 114.2 km
STARLINK-4734 | LEO | 122.1 km
STARLINK-40103 | LEO | 131.0 km
STARLINK-1454 | LEO | 132.5 km
STARLINK-1704 | LEO | 133.8 km
STARLINK-32257 | LEO | 134.9 km
YAOGAN-35 02B | LEO | 143.2 km
STARLINK-4458 | LEO | 152.5 km
STARLINK-4389 | LEO | 156.7 km


El más bajo es STARLINK-4734, a 113.1 km km. El más alto es TESS, a 283862.3 km.

#6 Comparación de grupos:
Altitud mínima, media y máxima de cada grupo.

In [16]:
resumen = resumen_por_grupo(satelites)
for nombre_grupo, datos in resumen.items():
    print(nombre_grupo, datos)

Otros {'cantidad': 4069, 'minima': 114.203, 'media': 6467.367592528877, 'maxima': 283289.762}
GPS {'cantidad': 40, 'minima': 19798.044, 'media': 20200.9868, 'maxima': 21025.232}
Globalstar {'cantidad': 36, 'minima': 677.281, 'media': 1291.842472222222, 'maxima': 1832.614}
BeiDou {'cantidad': 55, 'minima': 21484.54, 'media': 27218.0088, 'maxima': 35974.387}
Iridium {'cantidad': 80, 'minima': 630.06, 'media': 777.225325, 'maxima': 807.398}
OneWeb {'cantidad': 651, 'minima': 455.913, 'media': 1209.2284761904762, 'maxima': 1251.793}
Starlink {'cantidad': 11096, 'minima': 110.236, 'media': 467.9881021989906, 'maxima': 595.899}
Qianfan {'cantidad': 257, 'minima': 786.417, 'media': 1043.8097937743191, 'maxima': 1227.582}
Kuiper {'cantidad': 391, 'minima': 448.982, 'media': 632.0026547314578, 'maxima': 658.059}


Starlink tiene la altitud media más baja y BeiDou la más alta.

#7 Bariloche:
Satélites sobre la zona de Bariloche, de menor a mayor altitud.

In [17]:
BARILOCHE = (-76.30, -46.13, -66.30, -36.13)
sobre_bariloche = en_zona(satelites, BARILOCHE)
print("Satélites sobre Bariloche:", len(sobre_bariloche))
for s in ordenar_por(sobre_bariloche, "altitud_km"):
    print(s)

Satélites sobre Bariloche: 47
STARLINK-11548 [DTC] | LEO | 331.0 km
STARLINK-11595 [DTC] | LEO | 348.8 km
STARLINK-11533 [DTC] | LEO | 353.3 km
STARLINK-11647 [DTC] | LEO | 366.7 km
STARLINK-11174 [DTC] | LEO | 373.7 km
STARLINK-11398 [DTC] | LEO | 374.9 km
STARLINK-30172 | LEO | 381.5 km
STARLINK-3297 | LEO | 472.3 km
STARLINK-35312 | LEO | 473.4 km
STARLINK-3590 | LEO | 474.3 km
STARLINK-34549 | LEO | 474.5 km
STARLINK-35454 | LEO | 474.6 km
STARLINK-4266 | LEO | 476.1 km
STARLINK-5227 | LEO | 477.5 km
STARLINK-32194 | LEO | 478.3 km
STARLINK-5268 | LEO | 478.3 km
STARLINK-4436 | LEO | 480.8 km
STARLINK-4810 | LEO | 486.3 km
STARLINK-5518 | LEO | 489.7 km
STARLINK-33874 | LEO | 490.3 km
STARLINK-36258 | LEO | 490.4 km
STARLINK-34083 | LEO | 491.7 km
STARLINK-33942 | LEO | 492.2 km
STARLINK-34100 | LEO | 492.3 km
STARLINK-36468 | LEO | 492.6 km
STARLINK-36771 | LEO | 493.2 km
STARLINK-36448 | LEO | 494.4 km
STARLINK-35440 | LEO | 494.4 km
STARLINK-31611 | LEO | 494.5 km
STARLINK-35535

Ahora mismo hay 40 satélites sobre Bariloche.

#8 Conclusiones:
En este análisis observamos que el grupo predominante es Starlink, abarcando aproximadamente el 66.6% del total de satélites activos. El año con mayor cantidad de lanzamientos fue 2025. En cuanto a las trayectorias, la órbita LEO es la que predomina ampliamente, y analizando nuestra ubicación actual, se calculó que hay 40 satélites sobre Bariloche en este momento.